<a href="https://colab.research.google.com/github/roshjaison03/Transformer-model-from-scratch-for-cpu-only/blob/main/Transformer_using_SWIglu_%26_RoPe_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
del model

# **Took reference from** https://github.com/FareedKhan-dev/create-million-parameter-llm-from-scratch.git

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
import numpy as np
from matplotlib import pyplot as plt
import time
import pandas as pd
import urllib.request

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

# **We will be adding hyperparameters inside MASTER_CONGIG after each important architectural blocks**

In [ ]:
MASTER_CONGIG = {}

# **Dataset is a mix of Mark Twain books**

In [ ]:
lines = open('Test_dataset.txt','r').read()
sorted_lines = sorted(list(set(lines)))
print('Printing the first 10 characters of the vocab list:', sorted_lines[:10])

print('Total number of characters in our dataset (Vocabulary Size):', len(sorted_lines))

Printing the first 10 characters of the vocab list: ['\n', ' ', '!', '"', '$', '%', '&', "'", '(', ')']
Total number of characters in our dataset (Vocabulary Size): 92


In [ ]:
stoi = {character : index for index,character in enumerate(sorted_lines)}

itos = {index:character for index,character in enumerate(sorted_lines)}

In [ ]:
list(stoi.keys())[:10]
list(stoi.items())[:10]

[('\n', 0),
 (' ', 1),
 ('!', 2),
 ('"', 3),
 ('$', 4),
 ('%', 5),
 ('&', 6),
 ("'", 7),
 ('(', 8),
 (')', 9)]

In [ ]:
list(itos.keys())[:10]
list(itos.items())[:10]

[(0, '\n'),
 (1, ' '),
 (2, '!'),
 (3, '"'),
 (4, '$'),
 (5, '%'),
 (6, '&'),
 (7, "'"),
 (8, '('),
 (9, ')')]

In [ ]:
def encode(x):
  return [stoi[ch] for ch in x]

encode('morning')

output: [69, 71, 74, 70, 65, 70, 63] **bold text**

In [ ]:
def decode(l):
    return ''.join([itos[i] for i in l])

decode([69, 71, 74, 70, 65, 70, 63])

output = ['m', 'o', 'r', 'n', 'i', 'n', 'g']

In [ ]:
dataset = torch.tensor(encode(lines),dtype=torch.int8)
dataset.shape

torch.Size([1104041])

# **For Test purposes we are taking 1 Million charater dataset**

## torch.Size([1104041])

In [ ]:
MASTER_CONGIG = {
    "vocab_size":len(sorted_lines)
}

In [ ]:
def get_batches(data,split,batch_size,context_window,config=MASTER_CONGIG):
  train = data[:int(.8*len(data))]
  val = data[int(.8*len(data)):int(.9*len(data))]
  test = data[int(.9*len(data)):]

  batch_data = train
  if split == 'val':
    batch_data = val
  if split == 'test':
    batch_data = test

  ix = torch.randint(0,batch_data.size(0) - context_window -1,(batch_size,))

  x = torch.stack([batch_data[i:i+context_window]for i in ix]).long()
  y = torch.stack([batch_data[i+1:i+context_window+1]for i in ix]).long()
  x = x.to(device)
  y = y.to(device)
  return x,y

In [ ]:
MASTER_CONGIG.update({
    'batch_size': 128,
    'context_window': 128
})

In [ ]:
xs,ys = get_batches(dataset,'train',MASTER_CONGIG['batch_size'],MASTER_CONGIG['context_window'])
x = xs.to(device)
y = ys.to(device)
decoded_samples = [(decode(xs[i].tolist()), decode(ys[i].tolist())) for i in range(len(xs))]
decoded_samples[:10]

[('. Imagine the size of the\nsilence that would result on the instant. And imagine the feelings of\nthose bald-heads, and the exulta',
  ' Imagine the size of the\nsilence that would result on the instant. And imagine the feelings of\nthose bald-heads, and the exultat'),
 ('\nsuccessors; for we used to swim out a quarter or third of a mile and get\non these rafts and have a ride.\n\nBy way of illustratin',
  'successors; for we used to swim out a quarter or third of a mile and get\non these rafts and have a ride.\n\nBy way of illustrating'),
 ('ly displaying the sentence set forth in paragraph 1.E.1 with\nactive links or immediate access to the full terms of the Project\nG',
  'y displaying the sentence set forth in paragraph 1.E.1 with\nactive links or immediate access to the full terms of the Project\nGu'),
 ('I do not mean that it has constituted me a\njudge of men--no, it has not done that; for judges of men are born, not\nmade. My prof',
  ' do not mean that it has constituted me

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_mult=4):
        super().__init__()
        hidden_dim = hidden_mult * d_model

        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x):
        return self.w3(
            self.w1(x) * F.silu(self.w2(x))
        )


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        # x: (B, T, D)
        rms = torch.sqrt(
            torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps
        )
        x_norm = x / rms
        return x_norm * self.scale


In [ ]:
class RoPEMaskedAttentionHead(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.d_model = config["d_model"]
        self.context_window = config["context_window"]

        self.w_q = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_k = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_v = nn.Linear(self.d_model, self.d_model, bias=False)

        # Precompute RoPE frequencies
        self.register_buffer(
            "inv_freq",
            1.0 / (10000 ** (torch.arange(0, self.d_model, 2).float() / self.d_model))
        )

    def apply_rope(self, x):
        """
        x: (B, T, D)
        """
        B, T, D = x.shape

        pos = torch.arange(T, device=x.device).type_as(self.inv_freq)
        angles = torch.einsum("t,d->td", pos, self.inv_freq)

        sin = angles.sin()[None, :, :]
        cos = angles.cos()[None, :, :]

        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]

        x_rotated = torch.cat(
            [
                x_even * cos - x_odd * sin,
                x_even * sin + x_odd * cos,
            ],
            dim=-1,
        )

        return x_rotated

    def forward(self, x, return_attn_weights=False):
        """
        x: (B, T, D)
        """
        B, T, D = x.shape

        q = self.apply_rope(self.w_q(x))
        k = self.apply_rope(self.w_k(x))
        v = self.w_v(x)

        attn_output = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True,
            dropout_p=0.1 if self.training else 0.0,
        )

        if return_attn_weights:
            scores = torch.matmul(q, k.transpose(-2, -1)) / (D ** 0.5)
            mask = torch.tril(torch.ones(T, T, device=x.device))
            scores = scores.masked_fill(mask == 0, float("-inf"))
            attn_weights = F.softmax(scores, dim=-1)
            return attn_output, attn_weights

        return attn_output


In [ ]:
class RoPEMaskedMultiheadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.d_model = config['d_model']
        self.n_heads = config['n_heads']
        self.head_dim = self.d_model // self.n_heads

        assert self.d_model % self.n_heads == 0

        self.w_q = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_k = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_v = nn.Linear(self.d_model, self.d_model, bias=False)

        self.out_proj = nn.Linear(self.d_model, self.d_model)
        self.dropout = nn.Dropout(0.1)

        # RoPE frequencies (per head_dim)
        self.register_buffer(
            "inv_freq",
            1.0 / (10000 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        )

    def apply_rope(self, x):
        # x: (B, n_heads, T, head_dim)
        B, H, T, D = x.shape

        pos = torch.arange(T, device=x.device).type_as(self.inv_freq)
        angles = torch.einsum("t,d->td", pos, self.inv_freq)

        sin = angles.sin()[None, None, :, :]
        cos = angles.cos()[None, None, :, :]

        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]

        x_rot = torch.cat(
            [x_even * cos - x_odd * sin,
             x_even * sin + x_odd * cos],
            dim=-1
        )

        return x_rot

    def forward(self, x):
        # x: (B, T, d_model)
        B, T, _ = x.shape

        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)

        # reshape → (B, n_heads, T, head_dim)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # apply RoPE to Q and K
        q = self.apply_rope(q)
        k = self.apply_rope(k)

        # scaled dot-product attention
        attn = F.scaled_dot_product_attention(
            q, k, v,
            is_causal=True,
            dropout_p=0.1 if self.training else 0.0
        )

        # concat heads → (B, T, d_model)
        attn = attn.transpose(1, 2).contiguous()
        attn = attn.view(B, T, self.d_model)

        return self.dropout(self.out_proj(attn))


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.rms1 = RMSNorm(config['d_model'])

        self.attn = RoPEMaskedMultiheadAttention(config)

        self.rms2 = RMSNorm(config['d_model'])

        self.mlp = SwiGLU(config['d_model'], hidden_mult=4)


    def forward(self, x):
        # Attention block
        x = x + self.attn(self.rms1(x))

        # Feed-forward block
        x = x + self.mlp(self.rms2(x))

        return x


In [ ]:
class simple_test_model(nn.Module):
    def __init__(self, config=MASTER_CONGIG):
        super().__init__()
        self.config = config

        self.embedding = nn.Embedding(
            config['vocab_size'],
            config['d_model']
        )

        # STACK OF TRANSFORMER LAYERS
        self.blocks = nn.ModuleList([
            TransformerBlock(config)
            for _ in range(config['n_layers'])
        ])

        self.final_rms = RMSNorm(
            (config['d_model'])
        )

        self.lm_head = nn.Linear(
            config['d_model'],
            config['vocab_size']
        )

        print("Model parameters:",
              sum(p.numel() for p in self.parameters()))

    def forward(self, idx, targets=None):
        x = self.embedding(idx)

        for block in self.blocks:
            x = block(x)

        x = self.final_rms(x)
        logits = self.lm_head(x)

        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.config['vocab_size']),
                targets.view(-1)
            )
            return logits, loss

        return logits


In [ ]:
MASTER_CONGIG.update({
    'd_model': 128,
    'n_heads': 8,
    'n_layers':8,
})

In [ ]:
MASTER_CONGIG.update({
    'epochs': 50000,
    'log_interval': 1000,
    'batch_size': 128,
})

In [ ]:
MASTER_CONGIG

{'vocab_size': 92,
 'batch_size': 128,
 'context_window': 128,
 'd_model': 128,
 'n_heads': 8,
 'n_layers': 8,
 'epochs': 50000,
 'log_interval': 1000}

In [ ]:
model = simple_test_model(MASTER_CONGIG).to(device)
xs, ys = get_batches(dataset, 'train', MASTER_CONGIG['batch_size'], MASTER_CONGIG['context_window'])
xs = xs.to(device)
ys = ys.to(device)
logits, loss = model(xs, ys)

Model parameters: 2123996


In [ ]:
model = simple_test_model(MASTER_CONGIG).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.01
)

Model parameters: 2123996


In [ ]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10_000
)


In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, path="checkpoint.pt"):
    torch.save({
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict() if scheduler else None,
        "epoch": epoch,
    }, path)


In [ ]:
def load_checkpoint(model, optimizer, scheduler, path, device):
    checkpoint = torch.load(path, map_location=device)

    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])

    if scheduler and checkpoint["scheduler_state"] is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state"])

    start_epoch = checkpoint.get("epoch", 0)
    return start_epoch


In [ ]:
def continue_train(
    model,
    optimizer,
    scheduler,
    start_epoch,
    config=MASTER_CONGIG,
    print_logs=True,
):
    losses = []
    start_time = time.time()

    model.train()

    for epoch in range(start_epoch, config['epochs']):
        optimizer.zero_grad()

        xs, ys = get_batches(
            dataset,
            'train',
            config['batch_size'],
            config['context_window']
        )

        logits, loss = model(xs, targets=ys)
        loss.backward()
        optimizer.step()

        if scheduler:
            scheduler.step()

        if epoch % config['log_interval'] == 0:
            batch_time = time.time() - start_time
            x = evaluate_loss(model)
            losses.append(x)

            if print_logs:
                print(
                    f"Epoch {epoch} | "
                    f"train loss {x['train']:.3f} | "
                    f"val loss {x['val']:.3f} | "
                    f"Time {batch_time:.3f} | "
                    f"ETA {(batch_time * (config['epochs'] - epoch) / config['log_interval']):.3f}"
                )

            start_time = time.time()

            if scheduler:
                print("lr:", scheduler.get_last_lr())

            # 🔥 SAVE CHECKPOINT HERE
            save_checkpoint(
                model,
                optimizer,
                scheduler,
                epoch,
                path="checkpoint.pt"
            )

    print("Final Validation loss:", losses[-1]['val'])
    return pd.DataFrame(losses).plot()


In [ ]:
@torch.no_grad()
def evaluate_loss(model,config=MASTER_CONGIG):
  out = {}
  model.eval()
  for split in ["train","val"]:
    losses = []
    for _ in range(10):
      xy,yb = get_batches(dataset, split, MASTER_CONGIG['batch_size'], MASTER_CONGIG['context_window'])
      _,loss = model(xy,yb)
      losses.append(loss.item())
    out[split] = np.mean(losses)
  model.train()
  return out

In [ ]:
MASTER_CONGIG

{'vocab_size': 92,
 'batch_size': 128,
 'context_window': 128,
 'd_model': 128,
 'n_heads': 8,
 'n_layers': 8,
 'epochs': 50000,
 'log_interval': 1000}

In [ ]:
def train(model, optimizer, scheduler, config=MASTER_CONGIG, print_logs=True):
    losses = []

    start_time = time.time()

    for epoch in range(config['epochs']):
        optimizer.zero_grad()
        xs, ys = get_batches(dataset, 'train', config['batch_size'], config['context_window'])
        logits, loss = model(xs, targets=ys)
        loss.backward()
        optimizer.step()
        if scheduler:
            scheduler.step()

        if epoch % config['log_interval'] == 0:
            batch_time = time.time() - start_time
            x = evaluate_loss(model)
            losses += [x]

            if print_logs:
                print(f"Epoch {epoch} |train loss {x['train']:.3f} |val loss {x['val']:.3f} | Time {batch_time:.3f} | ETA in seconds {batch_time * (config['epochs'] - epoch)/config['log_interval'] :.3f}")

            start_time = time.time()

            if scheduler:
                print("lr: ", scheduler.get_lr())

    print("Validation loss: ", losses[-1]['val'])
    return pd.DataFrame(losses).plot()

train(model, optimizer,scheduler)

Epoch 0 |train loss 4.391 |val loss 4.395 | Time 0.544 | ETA in seconds 27.197
lr:  [0.00029999998519559363]


/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:1090: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)


Epoch 1000 |train loss 1.386 |val loss 1.553 | Time 147.193 | ETA in seconds 7212.457
lr:  [0.0002926293399246247]
Epoch 2000 |train loss 1.215 |val loss 1.468 | Time 151.979 | ETA in seconds 7294.972
lr:  [0.00027129714255383494]
Epoch 3000 |train loss 1.121 |val loss 1.424 | Time 152.157 | ETA in seconds 7151.363
lr:  [0.0002380915371919154]
Epoch 4000 |train loss 1.041 |val loss 1.427 | Time 151.974 | ETA in seconds 6990.813
lr:  [0.00019626291984859811]
Epoch 5000 |train loss 0.978 |val loss 1.429 | Time 152.207 | ETA in seconds 6849.321
lr:  [0.00014990576702634875]
Epoch 6000 |train loss 0.931 |val loss 1.434 | Time 151.972 | ETA in seconds 6686.753
lr:  [0.00010355783983234435]
Epoch 7000 |train loss 0.892 |val loss 1.410 | Time 152.138 | ETA in seconds 6541.936
lr:  [6.175599630466276e-05]
Epoch 8000 |train loss 0.869 |val loss 1.410 | Time 152.119 | ETA in seconds 6389.016
lr:  [2.859209213661384e-05]
Epoch 9000 |train loss 0.850 |val loss 1.445 | Time 152.076 | ETA in seconds

KeyboardInterrupt: 

In [ ]:
del model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = simple_test_model(MASTER_CONGIG).to(device)

ckpt = torch.load("/content/model_50k_8layer_context-window16.pt", map_location=device)
model.load_state_dict(ckpt["model_state"])

model.eval()



In [ ]:
save_checkpoint(model, optimizer, scheduler, epoch=0, path='model_context-window128.pt')
print("Model saved successfully to model_50k_8layer_context-window16.pt")

Model saved successfully to model_50k_8layer_context-window16.pt


In [ ]:
#del model

In [ ]:
MASTER_CONGIG.update({
    'context_window':16
})

In [ ]:
import time
import torch
import torch.nn.functional as F

@torch.no_grad()
def generate(
    model,
    prompt="The river was quiet",
    max_new_tokens=1000,
    temperature=0.8,
    top_k=40,
):
    model.eval()
    device = next(model.parameters()).device

    # encode prompt
    idx = torch.tensor(
        [encode(prompt)],
        dtype=torch.long,
        device=device
    )

    start_time = time.time()

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -MASTER_CONGIG['context_window']:]

        logits = model(idx_cond)
        logits = logits[:, -1, :] / temperature

        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float("inf")

        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)

        idx = torch.cat([idx, idx_next], dim=1)

    elapsed = time.time() - start_time

    text = decode(idx[0].tolist())
    text = text.replace("\\n", " ")
    text = " ".join(text.split())

    tokens_generated = idx.shape[1] - len(encode(prompt))
    tokens_per_sec = tokens_generated / max(elapsed, 1e-8)

    return {
        "text": text,
        "time_sec": elapsed,
        "tokens": tokens_generated,
        "tokens_per_sec": tokens_per_sec,
    }


In [ ]:
out = generate(model, prompt="The river was quiet")

print(out["text"])
print(f"\nTime: {out['time_sec']:.2f}s")
print(f"Tokens: {out['tokens']}")
print(f"Speed: {out['tokens_per_sec']:.2f} tokens/sec")


The river was quiet--a robber called 'The Employing any Commontean to the stage and sick or ten days. The first law as in six hundred and fifty years ago do not support the perfect day. It is sistered to the clusters of the park, in the old foreigner that the chief man would suplicate the best of four feet. When I find that was all there in the death-room, and he kept the most streams an eye that checks this and the date of the canoe that was drawing out of the streets, and the church from the forty feet of or lost some river from the amount the hotel. It was a slight whence he was so strange off to the monster's pluck, then. The other side of the leaf the letter--a matter--was petrified to care the monument that one night they were all killed; and a man that had to take out of it to any boat that they wanted to consider them. The present genuine South came and help is one of this boat is also at the cattle of the water was not have roosting further along. A by-loom hot does the captai

# **------------------------------FINETUNING------------------------------------**

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("Amod/mental_health_counseling_conversations")

In [ ]:
all_chars_from_dataset = set()
for split in ds:
    for example in ds['train']:
        # Concatenate all string fields from the example
        full_text = ""
        for key, value in example.items():
            if isinstance(value, str):
                full_text += value
        all_chars_from_dataset.update(list(full_text))

# Combine with existing sorted_lines to be safe, or just use new ones
# For fine-tuning, the tokenizer should probably reflect the fine-tuning data.
# Let's completely rebuild based on the new dataset.
sorted_chars_from_ds = sorted(list(all_chars_from_dataset))
print('New sorted characters (first 10):', sorted_chars_from_ds[:10])
print('New vocabulary size:', len(sorted_chars_from_ds))

# stoi = {character : index for index,character in enumerate(sorted_chars_from_ds)}
# itos = {index:character for index,character in enumerate(sorted_chars_from_ds)}

print(f"vocab size is {len(sorted_chars_from_ds)}")
# print('Updated MASTER_CONGIG vocab_size:', MASTER_CONGIG['vocab_size'])

# tokenized_train = ds["train"].map(
#     tokenize_example,
#     remove_columns=ds["train"].column_names
# )

New sorted characters (first 10): ['\n', '\r', ' ', '!', '"', '#', '$', '%', '&', "'"]
New vocabulary size: 112
vocab size is 112


In [ ]:
print(f"Length of stoi: {len(stoi)}")
print(f"Length of sorted_chars_from_ds: {len(sorted_chars_from_ds)}")

is_identical = (sorted_chars_from_ds == list(stoi.keys()))
print(f"Are sorted_chars_from_ds and the keys of stoi identical? {is_identical}")

if not is_identical:
    missing_in_stoi = set(sorted_chars_from_ds) - set(stoi.keys())
    missing_in_sorted_chars_from_ds = set(stoi.keys()) - set(sorted_chars_from_ds)
    if missing_in_stoi: print(f"Characters in sorted_chars_from_ds but not in stoi: {sorted(list(missing_in_stoi))}")
    if missing_in_sorted_chars_from_ds: print(f"Characters in stoi but not in sorted_chars_from_ds: {sorted(list(missing_in_sorted_chars_from_ds))}")

Length of stoi: 92
Length of sorted_chars_from_ds: 112
Are sorted_chars_from_ds and the keys of stoi identical? False
Characters in sorted_chars_from_ds but not in stoi: ['\r', '#', '+', '<', '=', '>', '@', '~', '\xa0', '¡', '·', '¿', 'Ú', 'á', 'é', 'í', 'ñ', 'ó', 'ú', 'ü', '–', '…']
Characters in stoi but not in sorted_chars_from_ds: ['•', '™']


In [ ]:
SYSTEM_PROMPT = "{System}: You are an empathetic listener who understands emotions and responds calmly, thoughtfully, and supportively.\n\n"

In [ ]:
def format_example(example):
    system = "You are an empathetic listener who understands emotions and responds calmly, thoughtfully, and supportively."

    user = example.get("Context", "").strip()
    assistant = example.get("Response", "").strip()

    prompt = (
        f"System: {system}\n"
        f"User: {user}\n"
        f"Assistant: "
    )

    return prompt, assistant


In [ ]:
def filter_text_by_vocab(text, stoi):
    """
    Removes all characters from `text` that are not present in `stoi`.

    text: str
    stoi: dict or iterable of allowed characters

    returns: filtered text (str)
    """
    vocab_set = set(stoi.keys())
    return "".join(ch for ch in text if ch in vocab_set)


In [ ]:
def tokenize_example(example):
    prompt, answer = format_example(example)

    full_text = prompt + answer

    # remove unsupported characters
    full_text = filter_text_by_vocab(full_text, stoi)

    tokens = encode(full_text)

    prompt_tokens = encode(filter_text_by_vocab(prompt, stoi))
    prompt_len = len(prompt_tokens)

    input_ids = tokens[:-1]
    targets = tokens[1:]

    # 🔥 Mask everything before Assistant response
    targets[:prompt_len - 1] = [-100] * (prompt_len - 1)

    return {
        "input_ids": input_ids,
        "targets": targets
    }


tokenized_train = ds["train"].map(
    tokenize_example,
    remove_columns=ds["train"].column_names
)

In [ ]:
import random
import torch

def get_batches_from_tokenized(dataset, batch_size, context_window, device):
    xs, ys = [], []

    for _ in range(batch_size):
        sample = random.choice(dataset)

        tokens = sample["input_ids"]
        targets = sample["targets"]

        if len(tokens) <= context_window:
            continue

        start = random.randint(0, len(tokens) - context_window - 1)

        x = tokens[start : start + context_window]
        y = targets[start : start + context_window]

        xs.append(x)
        ys.append(y)

    xs = torch.tensor(xs, dtype=torch.long, device=device)
    ys = torch.tensor(ys, dtype=torch.long, device=device)

    return xs, ys


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,        # 🔥 very important
    betas=(0.9, 0.95)
)


In [ ]:
num_steps = 5000

In [ ]:
for step in range(num_steps):
    xs, ys = get_batches_from_tokenized(
        tokenized_train,
        batch_size=MASTER_CONGIG["batch_size"],
        context_window=MASTER_CONGIG["context_window"],
        device=device
    )

    logits, loss = model(xs, targets=ys)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % 500 == 0:
        print(f"step {step} | loss {loss.item():.4f}")

step 0 | loss 1.0229
step 500 | loss 1.0722
step 1000 | loss 1.0101
step 1500 | loss 1.0225
step 2000 | loss 0.9839
step 2500 | loss 1.0129
step 3000 | loss 0.9936
step 3500 | loss 0.9663
step 4000 | loss 0.9971
step 4500 | loss 0.9873


In [ ]:
prompt = (
    "System: You are an empathetic listener who understands emotions "
    "and responds calmly, thoughtfully, and supportively.\n\n"
    "User: I am feeling very anxious about my upcoming exams. I can't seem to focus."
    "Assistant:"
)

out = generate(model, prompt=prompt)
print(out["text"])

System: You are an empathetic listener who understands emotions and responds calmly, thoughtfully, and supportively. User: I am feeling very anxious about my upcoming exams. I can't seem to focus.Assistant: The most important to you, I recommend a local and anger problem. What do you talk about what to large moments while over themselves in the moments and do you have always don't suggest that someone else is a lot of you work is to give you a specific time to be called out and asking your experiences of anxiety days to notice the same circumstance at the amount of the positions of your writing with your friend, then you may need to understand what you want to stick to yourself that there is in a body are making questions that is both can move out of her. Good between sweet relationships will ask him more about their past, you can know what you have been doing is in other crisis only on their reaction that people. There is no habitation that there is always encourage you to seek out th

In [ ]:
save_checkpoint(model, optimizer, scheduler, epoch=0, path='emotional_model_12ksteps.pt')
print("Model saved successfully to emotional_model_10ksteps.pt")

Model saved successfully to emotional_model_10ksteps.pt
